In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive


def run(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)


drive.mount("/content/drive")

WORKDIR = Path("/content/drive/MyDrive/Research/SSM_World_Models")
R2DREAMER_DIR = WORKDIR / "r2dreamer"
WORKDIR.mkdir(parents=True, exist_ok=True)

repo = "https://github.com/hanswalker/r2dreamer.git"
branch = "main"
if R2DREAMER_DIR.exists():
    run(["git", "remote", "set-url", "origin", repo], cwd=R2DREAMER_DIR)
    run(["git", "fetch", "origin"], cwd=R2DREAMER_DIR)
    run(["git", "checkout", "-B", branch, f"origin/{branch}"], cwd=R2DREAMER_DIR)
    run(["git", "reset", "--hard", f"origin/{branch}"], cwd=R2DREAMER_DIR)
else:
    run(["git", "clone", "--branch", branch, repo, str(R2DREAMER_DIR)])

os.chdir(R2DREAMER_DIR)
sys.path.insert(0, str(R2DREAMER_DIR))

from dmc_expert_colab import setup_colab

TDMPC2_DIR, DATA_DIR, Mamba3 = setup_colab(WORKDIR, R2DREAMER_DIR)


In [ ]:
from pathlib import Path
import shutil
import time

from dmc_expert_collection import (
    load_collection_config,
    make_collect_config,
    read_progress,
    start_collector,
    sync_dataset,
)
from IPython.display import clear_output

SCENARIOS = [
    {"name": "cartpole", "task": "cartpole/swingup"},
    {"name": "walker", "task": "walker/walk"},
    {"name": "humanoid", "task": "humanoid/run"},
]

cfg = load_collection_config(
    R2DREAMER_DIR / "configs" / "dmc_expert_collection.yaml",
    tdmpc2_dir=TDMPC2_DIR,
    data_dir=DATA_DIR,
)

local_root = Path(cfg["local_output_dir"] or cfg["output_dir"])
collector = R2DREAMER_DIR / "scripts" / "collect_dmc_expert_data.py"


def short(path):
    path = Path(path)
    for root in (WORKDIR, local_root):
        try:
            return str(path.relative_to(root))
        except ValueError:
            pass
    return str(path)


jobs = []
resume = bool(cfg["resume"])
for scenario in SCENARIOS:
    out_dir = local_root / scenario["name"]
    final_dir = Path(cfg["output_dir"]) / scenario["name"]

    if resume:
        if not out_dir.exists() and final_dir.exists():
            shutil.copytree(final_dir, out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
    else:
        for folder in (out_dir, final_dir):
            if folder.exists():
                shutil.rmtree(folder)
        out_dir.mkdir(parents=True, exist_ok=True)

    config_path = out_dir / "collect_config.yaml"
    log_path = out_dir / "collector.log"
    make_collect_config(cfg, scenario["task"], out_dir, config_path)
    proc, log_file = start_collector(collector, config_path, log_path)
    jobs.append({
        "scenario": scenario,
        "out_dir": out_dir,
        "final_dir": final_dir,
        "log": log_path,
        "proc": proc,
        "log_file": log_file,
    })

while True:
    clear_output(wait=True)
    print(f"Data collection: {len(jobs)} tasks x {cfg['num_episodes']} target episodes, resume={resume}")
    print(f"{'task':<18} {'status':<10} {'episodes':>11} {'rows':>10}  log")
    done = True

    for job in jobs:
        code = job["proc"].poll()
        status = "running" if code is None else ("done" if code == 0 else f"failed {code}")
        episodes, rows, _ = read_progress(job["out_dir"], job["scenario"]["task"])
        print(
            f"{job['scenario']['name']:<18} {status:<10} "
            f"{episodes:>4}/{int(cfg['num_episodes']):<6} {rows:>10}  {short(job['log'])}"
        )
        done = done and code is not None

    if done:
        break
    time.sleep(int(cfg["refresh_seconds"]))

failed = []
for job in jobs:
    job["log_file"].close()
    if job["proc"].returncode != 0:
        failed.append(job)

if failed:
    for job in failed:
        print()
        print(f"--- {job['scenario']['name']} last log lines ---")
        print(chr(10).join(job["log"].read_text(errors="replace").splitlines()[-80:]))
    raise RuntimeError(f"{len(failed)} collectors failed")

print("Syncing completed datasets to Drive...")
for job in jobs:
    final_dir = sync_dataset(job["out_dir"], job["final_dir"])
    print(f"{job['scenario']['name']:<18} -> {short(final_dir)}")

print("Collection finished.")


In [ ]:
import importlib
import dmc_expert_collection
import dmc_expert_training
import dmc_expert_evaluation

importlib.reload(dmc_expert_collection)
importlib.reload(dmc_expert_training)
importlib.reload(dmc_expert_evaluation)

In [ ]:
from pathlib import Path
import time

from dmc_expert_collection import task_store_name
from dmc_expert_training import read_metric_summary, start_training
from IPython.display import clear_output

MODELS = [
    {"name": "gru", "config": "offline_dmc_expert_gru"},
    {"name": "mamba3", "config": "offline_dmc_expert_mamba3"},
]

run_tag = time.strftime("%Y%m%d_%H%M%S")
RUNS = []
for scenario in SCENARIOS:
    for model in MODELS:
        RUNS.append({
            "name": f"{scenario['name']}_{model['name']}",
            "config": model["config"],
            "data": DATA_DIR / scenario["name"] / task_store_name(scenario["task"]),
            "task": "dmc_" + scenario["task"].replace("/", "_"),
            "logdir": WORKDIR / "runs" / f"offline_{scenario['name']}_{model['name']}_{run_tag}",
        })


def short(path):
    return str(Path(path).relative_to(WORKDIR))


jobs = []
for run in RUNS:
    proc, log_file, stdout_log = start_training(
        r2dreamer_dir=R2DREAMER_DIR,
        config=run["config"],
        data=run["data"],
        task=run["task"],
        logdir=run["logdir"],
        resume=False,
    )
    jobs.append({"run": run, "proc": proc, "log_file": log_file, "stdout": stdout_log})

while True:
    clear_output(wait=True)
    print("Offline training: GRU vs Mamba3")
    print(f"{'run':<18} {'status':<10} {'step':>8} {'loss':>10} {'eval':>10} {'ckpt':<9} log")
    done = True

    for job in jobs:
        run = job["run"]
        code = job["proc"].poll()
        status = "running" if code is None else ("done" if code == 0 else f"failed {code}")
        step, loss, score = read_metric_summary(run["logdir"])
        ckpt = "latest" if (run["logdir"] / "latest.pt").exists() else "-"
        ckpt = "best" if (run["logdir"] / "best.pt").exists() else ckpt
        print(f"{run['name']:<18} {status:<10} {step:>8} {loss:>10} {score:>10} {ckpt:<9} {short(job['stdout'])}")
        done = done and code is not None

    if done:
        break
    time.sleep(10)

failed = []
for job in jobs:
    job["log_file"].close()
    if job["proc"].returncode != 0:
        failed.append(job)

if failed:
    for job in failed:
        print()
        print(f"--- {job['run']['name']} last log lines ---")
        print(chr(10).join(job["stdout"].read_text(errors="replace").splitlines()[-80:]))
    raise RuntimeError(f"{len(failed)} training runs failed")

print("Training finished.")

In [ ]:
from dmc_expert_evaluation import eval_run

results = []
for run in RUNS:
    print(f"Evaluating {run['name']}...")
    result = eval_run(
        r2dreamer_dir=R2DREAMER_DIR,
        config=run["config"],
        data=run["data"],
        task=run["task"],
        logdir=run["logdir"],
        episodes=5,
        checkpoint="best.pt",
    )
    results.append((run, result))

print()
print("Eval summary")
for run, item in results:
    print(
        f"{run['name']:<18} fresh={item['fresh_eval_score']} "
        f"best_logged={item['logged_best_eval']} ckpt={item['checkpoint']}"
    )

results